# Smith-Waterman: C vs CUDA - setup and checks (Weeks 7-8)
CEPARCO Integrating Project, Group 3

Checks that the Colab environment works for the project: GPU, `nvcc`, a tiny CUDA
compile-and-run test, the repository, the test inputs, and (Week 8) the sequential C program.
Colab runs Linux; this notebook is only for the **final timings and CUDA work**. Day-to-day
development of the C code is done on Windows with `build.bat` (see the README).

**Before running:** Runtime -> Change runtime type -> choose a **GPU** (usually a T4).

## 1. GPU and compiler

In [ ]:
import os
os.environ["PATH"] += ":/usr/local/cuda/bin"      # in case nvcc is not already on PATH
!nvidia-smi
!nvcc --version
!gcc --version | head -1

## 2. CUDA smoke test
Compiles and runs a 1,000-thread kernel with the same flags the project will use
(`-arch=native` lets `nvcc` detect the GPU; needs CUDA 11.5 or newer - otherwise use e.g. `-arch=sm_75`).
Expected output: `sum=499500 (expected 499500) -> OK`

In [ ]:
%%writefile hello.cu
#include <cstdio>
__global__ void fill(int *x, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) x[i] = i;
}
int main() {
    const int n = 1000;
    int *d, h[1000];
    cudaMalloc(&d, n * sizeof(int));
    fill<<<(n + 255) / 256, 256>>>(d, n);
    cudaMemcpy(h, d, sizeof h, cudaMemcpyDeviceToHost);
    long s = 0;
    for (int i = 0; i < n; i++) s += h[i];
    printf("sum=%ld (expected 499500) -> %s\n", s, s == 499500 ? "OK" : "WRONG");
    cudaFree(d);
    return s == 499500 ? 0 : 1;
}

In [ ]:
!nvcc -O2 -arch=native hello.cu -o hello && ./hello

## 3. Get the repository
Clones the team repository. Edit `REPO_URL` and `BRANCH` first if they differ.
The cell deletes and re-clones `sw_repo` every time, so re-running it always gives a fresh copy.

In [ ]:
REPO_URL = "https://github.com/Luis-AQR/CEPARCO_GRP3_IntegratingProject.git"
BRANCH   = "main"          # use the branch you want to test, e.g. "build/week8-sequential-c"

%cd /content
!rm -rf sw_repo
!git clone --branch $BRANCH $REPO_URL sw_repo
%cd sw_repo
!ls

## 4. Validate the test sequences
Runs an independent pure-Python Smith-Waterman on the four small test inputs and compares
with the expected scores, coordinates, alignments and (for the worked example) the full
score matrix; also checks the format of the benchmark inputs.

In [ ]:
import os
# Works with the flattened layout (scripts/ at the repo root) and the old smith-waterman/ sub-folder.
root = "/content/sw_repo"
if not os.path.isdir(f"{root}/scripts") and os.path.isdir(f"{root}/smith-waterman/scripts"):
    root = f"{root}/smith-waterman"
%cd $root
!ls
!python3 scripts/check_expected.py

## 5. Benchmark inputs are reproducible
Regenerates each benchmark file from its seed and compares it with the committed copy.
All three lines should say `identical`.

In [ ]:
%%bash
# generator always writes LF, and the committed files are LF, so cmp must say "identical"
for n in 1000 5000 10000; do
  python3 scripts/gen_sequences.py --length $n --seed 1 --out /tmp/regen_$n.fasta > /dev/null
  if cmp -s /tmp/regen_$n.fasta data/bench_$n.fasta; then
    echo "bench_$n: identical"
  else
    echo "bench_$n: DIFFERENT (committed file is authoritative; tell the team)"
  fi
done

## 6. Record the environment
Colab's GPU depends on availability. Paste this line into the README / report so every
timing can be traced to the machine that produced it.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version | tail -1
!python3 --version

## 7. Week 8: build and run the sequential C program
Compiles `c/sw_seq.c` with gcc (Colab's compiler), prints the worked example, then compares the
C results with the independent Python reference on the test inputs and `bench_1000`
(score, best cell, matrix checksum, and the full matrix for the small tests).
Expected: every line `ok`, then `All C results match the reference.`

In [ ]:
%%bash
mkdir -p bin
gcc -std=c99 -O2 -Wall -Wextra -o bin/sw_seq c/sw_seq.c
./bin/sw_seq data/test_paper.fasta
echo
./bin/sw_seq data/bench_10000.fasta --no-matrix
echo
python3 scripts/check_c_week8.py

## 8. (Optional) include bench_5000 in the check
The pure-Python reference takes a while on 5,000 x 5,000, so this is a separate cell.

In [ ]:
!python3 scripts/check_c_week8.py --all